# MINav Offline RL Training on Kaggle
This notebook extracts your real-world ROS 2 bag and runs the full DINOv3 + Hindsight relabeling dataset pipeline, followed by TD3+BC offline RL training on 2x T4 GPUs.

## Setup Instructions
1. **Upload Dataset:** Make sure your `MINav_Dataset_20260919_174738` directory is uploaded as a Kaggle Dataset.
2. **Clone Repo:** The notebook will clone your `RL-Image-Goal-Navigation` repository if it's not already in the working directory.
3. **Accelerator:** Set the Kaggle Notebook Accelerator to **GPU T4 x2**.

In [ ]:
import subprocess
print("Installing dependencies...")
subprocess.run(["pip", "install", "pyarrow", "pandas", "pyyaml", "wandb", "tensorboard", "torch", "torchvision", "torchaudio", "mujoco"], check=True)
print("Dependencies installed.")

In [ ]:
import os
import sys
import shutil

# ---------------------------------------------------------
# USER CONFIGURATION
# ---------------------------------------------------------
REPO_DIR = '/kaggle/working/RL'
# Update this path to match your Kaggle dataset mount point
DATASET_PATH = '/kaggle/input/minav-dataset-20260921-154717-processed/MINav_Dataset_20260921_154717_processed'
RUN_NAME = 'kaggle_run'

# ---------------------------------------------------------
# TEST MODE SETTINGS
# ---------------------------------------------------------
TEST_MODE = True
MAX_FRAMES = 1000
TRAIN_STEPS = 100
FQE_EVERY = 50

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/sanmita13742/RL-Image-Goal-Navigation.git", REPO_DIR], check=True)
else:
    print("Repository exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
sys.path.append(REPO_DIR)
print("Repository ready at:", REPO_DIR)


In [ ]:
import os
import cv2
import csv
import json
import numpy as np
import pandas as pd
import shutil

RUN_DIR = os.path.join(REPO_DIR, "realworld", "runs", RUN_NAME)
EXPLORE_DIR = os.path.join(RUN_DIR, "exploration")

def prepare_dataset(src_dir, out_dir):
    print(f"Preparing dataset from {src_dir} to {out_dir}...")
    seg_dir = os.path.join(out_dir, "segment_000")
    rgb_dir = os.path.join(seg_dir, "rgb")
    os.makedirs(rgb_dir, exist_ok=True)
    
    src_csv = os.path.join(src_dir, "data.csv")
    df = pd.read_csv(src_csv)
    
    csv_path = os.path.join(seg_dir, "segment.csv")
    csv_file = open(csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        "trajectory_id", "global_step", "segment_step", "sim_time",
        "linear_vel_cmd", "lateral_vel_cmd", "angular_vel_cmd",
        "executed_linear_vel", "executed_lateral_vel", "executed_angular_vel",
        "pos_x", "pos_y", "yaw", "rgb_path", "depth_path",
        "safety_intervention"
    ])
    
    start_time = df['timestamp'].iloc[0]
    total_processed = 0
    
    for i, row in df.iterrows():
        if TEST_MODE and i >= MAX_FRAMES:
            break
        
        sim_time = (row['timestamp'] - start_time) / 1e9
        
        src_img = os.path.join(src_dir, row['image'])
        # The pipeline handles either .jpg or .png natively as long as it's consistent
        img_filename = f"{i:06d}.jpg"
        dst_img = os.path.join(rgb_dir, img_filename)
        shutil.copy2(src_img, dst_img)
        
        csv_writer.writerow([
            0, i, i, sim_time,
            row['vx'], row['vy'], row['wz'],
            row['vx'], row['vy'], row['wz'],
            '', '', '',
            f"rgb/{img_filename}", "",
            0
        ])
        total_processed = i
        if (i + 1) % 100 == 0:
            print(f"Prepared {i + 1} frames...")
            
    csv_file.close()
    
    meta = {
        "run_id": RUN_NAME,
        "total_steps_recorded": total_processed + 1,
        "segment_size": total_processed + 10,
        "num_segments": 1,
        "source": "realworld",
        "segments": [{
            "segment_id": "segment_000",
            "global_start": 0,
            "global_end": total_processed,
            "num_steps": total_processed + 1
        }]
    }
    with open(os.path.join(out_dir, "exploration_metadata.json"), 'w') as f:
        json.dump(meta, f)
        
    print(f"\n✓ Preparation complete! Total frames: {total_processed + 1}")
    print(f"Dataset saved in: {out_dir}")

if not os.path.exists(os.path.join(EXPLORE_DIR, "exploration_metadata.json")):
    prepare_dataset(DATASET_PATH, EXPLORE_DIR)
else:
    print(f"{EXPLORE_DIR} already exists. Skipping preparation.")


In [ ]:
# ---------------------------------------------------------
# Set up configurations for TEST_MODE or full run
# ---------------------------------------------------------
import yaml
os.makedirs(RUN_DIR, exist_ok=True)
CONFIG_DST = os.path.join(RUN_DIR, "config_used.yaml")

with open("realworld/configs/realworld_pipeline.yaml", "r") as f:
    config = yaml.safe_load(f)
    
if TEST_MODE:
    config["smoke"] = {
        "enabled": True,
        "max_frames": MAX_FRAMES,
        "training_steps": TRAIN_STEPS
    }
    config["training"]["total_gradient_steps"] = TRAIN_STEPS
    config["training"]["checkpoint_every"] = FQE_EVERY
    config["training"]["fqe_frequency"] = FQE_EVERY
    config["training"]["fqe_steps"] = 50
else:
    config["smoke"] = {"enabled": False}

with open(CONFIG_DST, "w") as f:
    yaml.dump(config, f)

print("Config prepared at:", CONFIG_DST)


In [ ]:
# ---------------------------------------------------------
# Build Final Dataset (DINOv3 Encodings + Hindsight Goals)
# ---------------------------------------------------------
cmd = [
    sys.executable, "realworld/scripts/build_dataset.py",
    "--config", CONFIG_DST,
    "--run-dir", RUN_DIR,
    "--device", "cuda"
]
if TEST_MODE:
    cmd.append("--smoke")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# ---------------------------------------------------------
# Shape Verification
# ---------------------------------------------------------
print("\n--- Verification ---")
processed_dir = os.path.join(RUN_DIR, config.get("hindsight", {}).get("output_subdir", "processed"))
dinov3_dir = os.path.join(processed_dir, "dinov3")
phi_vectors_path = os.path.join(dinov3_dir, "phi_vectors.npy")

if os.path.exists(phi_vectors_path):
    phi_vectors = np.load(phi_vectors_path)
    print(f"DINOv3 Embeddings Shape: {phi_vectors.shape} (Expected: [N, {config.get('hindsight', {}).get('phi_dimension', 384)}])")
else:
    print(f"Error: {phi_vectors_path} not found.")
    
hindsight_dir = os.path.join(processed_dir, "hindsight")
geom_path = os.path.join(hindsight_dir, "geometric_transitions.parquet")
if os.path.exists(geom_path):
    import pandas as pd
    df = pd.read_parquet(geom_path)
    print(f"Geometric Dataset Rows: {len(df)}")


In [ ]:
# ---------------------------------------------------------
# Train TD3+BC Offline RL
# ---------------------------------------------------------
cmd = [
    sys.executable, "realworld/scripts/train.py",
    "--config", CONFIG_DST,
    "--run-dir", RUN_DIR,
    "--device", "cuda"
]
if TEST_MODE:
    cmd.append("--smoke")
    
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
